In [2]:
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

# Base directory containing day subdirectories
base_dir = Path('~/Downloads/extracted_messages/').expanduser()

# Nested dict: symbol -> MPID -> count
symbol_mpid_counts = defaultdict(Counter)

# Loop through all day subdirectories
for day_dir in sorted(base_dir.iterdir()):
    if not day_dir.is_dir():
        continue
    
    print(f"Processing {day_dir.name}...")
    
    # Loop through all parquet files in the day directory
    for parquet_file in day_dir.glob('*.parquet'):
        df = pd.read_parquet(parquet_file)
        
        # Check if required columns exist
        if 'order_id' not in df.columns:
            continue
        
        # Filter for F orders (rows with valid order_id)
        f_orders = df[df['order_id'].notna()]
        
        # Skip if no F orders in this file
        if f_orders.empty:
            continue
        
        # Check if mpid and stock columns exist
        if 'mpid' not in df.columns or 'stock' not in df.columns:
            continue
        
        # Tally frequency by symbol and MPID
        for (stock, mpid), count in f_orders.groupby(['stock', 'mpid']).size().items():
            symbol_mpid_counts[stock][mpid] += count

print(f"\nProcessed {len(symbol_mpid_counts)} symbols")
print(f"Total AddOrderMPID messages: {sum(sum(c.values()) for c in symbol_mpid_counts.values()):,}")

Processing 20250201...
Processing 20250203...
Processing 20250204...
Processing 20250205...
Processing 20250206...
Processing 20250207...
Processing 20250208...
Processing 20250210...
Processing 20250211...
Processing 20250212...
Processing 20250213...
Processing 20250214...
Processing 20250215...
Processing 20250218...
Processing 20250219...
Processing 20250220...
Processing 20250221...
Processing 20250222...
Processing 20250224...
Processing 20250225...
Processing 20250226...
Processing 20250227...
Processing 20250228...
Processing 20250301...
Processing 20250303...
Processing 20250304...
Processing 20250305...
Processing 20250306...
Processing 20250307...
Processing 20250308...
Processing 20250310...
Processing 20250311...
Processing 20250312...
Processing 20250313...
Processing 20250314...
Processing 20250315...
Processing 20250317...
Processing 20250318...
Processing 20250319...
Processing 20250320...
Processing 20250321...
Processing 20250322...
Processing 20250324...
Processing 

In [3]:
# Get top 5 symbols by total F order frequency
symbol_totals = {symbol: sum(mpids.values()) for symbol, mpids in symbol_mpid_counts.items()}
top_5_symbols = sorted(symbol_totals.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 Symbols by F Order Frequency:")
print("=" * 60)

for rank, (symbol, total_count) in enumerate(top_5_symbols, 1):
    print(f"\n{rank}. {symbol}: {total_count:,} total F orders")
    print("-" * 40)
    
    # Get top 5 MPIDs for this symbol
    top_5_mpids = symbol_mpid_counts[symbol].most_common(5)
    for mpid_rank, (mpid, count) in enumerate(top_5_mpids, 1):
        pct = (count / total_count) * 100
        print(f"   {mpid_rank}. {mpid}: {count:,} ({pct:.1f}%)")

Top 10 Symbols by F Order Frequency:

1. QULL: 1,295,859 total F orders
----------------------------------------
   1. VIRT: 1,270,095 (98.0%)
   2. CDRG: 17,934 (1.4%)
   3. ETMM: 6,585 (0.5%)
   4. UBSS: 371 (0.0%)
   5. SOHO: 264 (0.0%)

2. MTUL: 983,515 total F orders
----------------------------------------
   1. VIRT: 965,597 (98.2%)
   2. CDRG: 16,644 (1.7%)
   3. UBSS: 274 (0.0%)
   4. ETMM: 268 (0.0%)
   5. SOHO: 243 (0.0%)

3. UCYB: 938,058 total F orders
----------------------------------------
   1. VIRT: 635,935 (67.8%)
   2. JSCA: 200,740 (21.4%)
   3. FLTG: 99,264 (10.6%)
   4. CDRG: 708 (0.1%)
   5. ETMM: 322 (0.0%)

4. USML: 808,615 total F orders
----------------------------------------
   1. VIRT: 795,134 (98.3%)
   2. CDRG: 12,126 (1.5%)
   3. UBSS: 386 (0.0%)
   4. ETMM: 244 (0.0%)
   5. SOHO: 239 (0.0%)

5. VYMI: 803,518 total F orders
----------------------------------------
   1. FLTG: 788,607 (98.1%)
   2. UBSS: 9,494 (1.2%)
   3. JPMS: 1,157 (0.1%)
   4. GSCO:

In [4]:
# Convert dict of dicts to DataFrame and export to parquet
rows = []
for symbol, mpid_dict in symbol_mpid_counts.items():
    for mpid, count in mpid_dict.items():
        rows.append({'symbol': symbol, 'mpid': mpid, 'count': count})

export_df = pd.DataFrame(rows)
export_df.to_parquet('symbol_mpid_counts.parquet', index=False)

print(f"Exported {len(export_df):,} rows to symbol_mpid_counts.parquet")
print(export_df.head(10))

Exported 246,227 rows to symbol_mpid_counts.parquet
  symbol  mpid  count
0      A  FLTG   3584
1      A  GSCO   1789
2      A  MAXM    146
3      A  WBPX   2108
4      A  JPMS    596
5      A  WCHV    598
6      A  SGAS    807
7      A  GTSM    120
8      A  MSCO    247
9      A  VIRT    432
